# Inspecting and preparing two registers

This notebook does the work described in
[Data readiness](data-readiness.md): it looks at two administrative registers
as they arrive, decides whether they can be linked at all, and turns them into
something a linkage method can compare.

Nothing here is specific to a linkage method. Whatever you do afterwards,
deterministic or probabilistic, you do it on the output of a notebook like this
one.

**What you will do**

1. Load both registers without letting the software reinterpret your data
2. Profile completeness: how much of each identifying field is actually there
3. Profile cardinality: how much each field can distinguish one person from another
4. Look for duplicates inside each register
5. Check the three structural prerequisites that a linkage library needs
6. Clean and standardise names, sex and nationality
7. Measure what the cleaning changed

## 0. Setup

The two files are the synthetic registers that ship with this toolkit: 30,000
records from a health-insurance register and 27,000 from a social-security
register, with 4,500 people known to appear in both. See
[the data documentation](../../data/README.md) for provenance and schema.

In [1]:
import unicodedata
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

# Works whether the notebook is run from its own folder or the repository root
DATA = Path("../../data")
if not DATA.exists():
    DATA = Path("data")

print("data directory:", DATA.resolve())

data directory: /Users/nicolas/Documents/repos/record-linkage-toolkit/data


## 1. Load the data as text

The single most common way to damage identifying data is to let a CSV reader
guess its types. Identifiers that look numeric get read as integers, losing
leading zeros; codes get turned into floats; blank strings and the literal word
`NA` get silently converted to the same thing.

Read every identifying column as a string, always.

In [2]:
fonasa = pd.read_csv(DATA / "fonasa_sample.csv", dtype=str, keep_default_na=True)
suseso = pd.read_csv(DATA / "suseso_sample.csv", dtype=str, keep_default_na=True)

print(f"fonasa: {fonasa.shape[0]:,} rows x {fonasa.shape[1]} columns")
print(f"suseso: {suseso.shape[0]:,} rows x {suseso.shape[1]} columns")
print()
print("columns:", list(fonasa.columns))

fonasa: 30,000 rows x 7 columns
suseso: 27,000 rows x 7 columns

columns: ['unique_id', 'nombre', 'ap1', 'ap2', 'sexo', 'nacionalidad', 'true_person_id']


In [3]:
fonasa.head(5)

,unique_id,nombre,ap1,ap2,sexo,nacionalidad,true_person_id
0,F0000001,NaN,RUJES,VERA,MUJER,CHILENA,1528363
1,F0000002,NICOLÁS ALBERTK,LOBOS,KING,HOMBRE,CHILENA,570155
2,F0000003,JOSEFA DANIELA,RINCON,GUTIERREZ,MUJER,CHILENA,1173391
3,F0000004,GUIVALDO MOISÉS,HOBRAN,NaN,HOMBRE,CHILENA,687581
4,F0000005,KAREN IVONNE,GODOY,HUESPED,MUJER,CHILENA,1771159


Two things are already visible in those five rows: names carry accents, and
some fields are empty. Both matter.

`true_person_id` is the ground truth: two records with the same value are the
same person. **You will not have this column in real work.** It is here because
the data is synthetic, and this toolkit uses it only to measure how well a
method performs. It is never an input to a linkage method, and it is never
compared or blocked on.

## 2. Completeness

A field you cannot see is a field you cannot compare. Before anything else,
find out how much of each identifying field is actually populated.

In [4]:
def completeness(df, name):
    n = len(df)
    out = pd.DataFrame({
        "non_missing": df.notna().sum(),
        "missing": df.isna().sum(),
    })
    out["pct_missing"] = (100 * out["missing"] / n).round(2)
    out.insert(0, "register", name)
    return out

comp = pd.concat([completeness(fonasa, "fonasa"), completeness(suseso, "suseso")])
comp

,register,non_missing,missing,pct_missing
unique_id,fonasa,30000,0,0.00
nombre,fonasa,28507,1493,4.98
ap1,fonasa,28527,1473,4.91
ap2,fonasa,28445,1555,5.18
sexo,fonasa,29053,947,3.16
nacionalidad,fonasa,28623,1377,4.59
true_person_id,fonasa,30000,0,0.00
unique_id,suseso,27000,0,0.00
nombre,suseso,25717,1283,4.75
ap1,suseso,25695,1305,4.83


Between 2% and 5% of each identifying field is missing in both registers, and
the missingness is spread across fields rather than concentrated in a few broken
rows. That is typical of administrative data and it has a direct consequence:
any rule that requires every field to be present will discard a slice of the
population before it starts.

The question that actually matters is not how often one field is missing, but
how many records have **enough** fields to be compared at all.

In [5]:
name_fields = ["nombre", "ap1", "ap2", "sexo", "nacionalidad"]

for name, df in [("fonasa", fonasa), ("suseso", suseso)]:
    n_present = df[name_fields].notna().sum(axis=1)
    dist = n_present.value_counts().sort_index()
    print(f"{name}: number of the 5 identifying fields present")
    for k, v in dist.items():
        print(f"  {k} field(s): {v:6,}  ({100*v/len(df):5.2f}%)")
    print(f"  all five present: {100*(n_present == 5).mean():.2f}%")
    print()

fonasa: number of the 5 identifying fields present
  1 field(s):      2  ( 0.01%)
  2 field(s):     68  ( 0.23%)
  3 field(s):    852  ( 2.84%)
  4 field(s):  4,929  (16.43%)
  5 field(s): 24,149  (80.50%)
  all five present: 80.50%

suseso: number of the 5 identifying fields present
  2 field(s):     30  ( 0.11%)
  3 field(s):    532  ( 1.97%)
  4 field(s):  4,154  (15.39%)
  5 field(s): 22,284  (82.53%)
  all five present: 82.53%



About four records in five carry all five identifying fields. The rest carry
four or fewer, and those are the records a strict rule silently drops.

Whether that is acceptable depends on **who** the incomplete records are. If
missingness is random, losing them costs precision. If the people with
incomplete records differ systematically from the rest, which is common in
administrative data, losing them introduces bias into whatever you publish.

## 3. Cardinality: how much can each field distinguish?

Completeness tells you whether a field is there. Cardinality tells you whether
it is worth anything.

A field that takes two values cannot separate two people from a crowd; it can
only rule out about half of it. A field with hundreds of thousands of distinct
values is highly discriminating. This is exactly the intuition that the
Fellegi-Sunter framework later makes numerical, but you can see it now.

In [6]:
def cardinality(df, name, fields):
    rows = []
    for col in fields:
        s = df[col].dropna()
        vc = s.value_counts()
        rows.append({
            "register": name,
            "field": col,
            "distinct_values": s.nunique(),
            "most_common": vc.index[0] if len(vc) else None,
            "most_common_pct": round(100 * vc.iloc[0] / len(s), 2) if len(vc) else None,
        })
    return pd.DataFrame(rows)

pd.concat([
    cardinality(fonasa, "fonasa", name_fields),
    cardinality(suseso, "suseso", name_fields),
]).reset_index(drop=True)

,register,field,distinct_values,most_common,most_common_pct
0,fonasa,nombre,20698,DEL,0.28
1,fonasa,ap1,9951,GONZÁLEZ,0.88
2,fonasa,ap2,8468,MUÑOZ,0.98
3,fonasa,sexo,2,HOMBRE,50.75
4,fonasa,nacionalidad,2,CHILENA,82.19
5,suseso,nombre,18532,JUAN CARLOS,0.29
6,suseso,ap1,9910,GONZALEZ,0.83
7,suseso,ap2,11160,GONZALEZ,0.84
8,suseso,sexo,2,HOMBRE,53.16
9,suseso,nacionalidad,2,CHILENA,76.45


The ordering is stark. Given name and the two surnames each take thousands of
distinct values; sex takes two, and nationality takes two, with one value
covering the large majority of records.

That does not make sex and nationality useless. Agreement on sex is weak
evidence on its own, but *disagreement* on sex is fairly strong evidence
against a pair, and both fields are cheap to compare. What it does mean is that
a rule built only on sex and nationality would return an enormous number of
pairs that agree by coincidence.

Two details in that table are worth pausing on, because they are the kind of
thing profiling exists to surface.

The most common first surname is `GONZÁLEZ` in one register and `GONZALEZ` in
the other. It is the same surname. As the data stands, those two values will
never compare as equal, and neither will any of the thousands of other accented
names. Nothing about the linkage method fixes this; the fix is cleaning, and it
has to be the *same* cleaning on both sides.

The most common given name in the first register is `DEL`, which is not a given
name at all but a fragment of a compound one. When a token like that rises to
the top of a frequency table, something upstream is splitting or truncating
values. Whether you can fix it or only work around it, you want to know before
you build a model on that field.

## 4. Duplicates within a register

Before asking whether a record in one register matches a record in the other,
ask how many records each register holds *per person*. If one register contains
the same person three times, every match you find will be a match to three
records, and any count you publish will be wrong.

There is no identifier to group by, so use the identifying fields themselves as
a rough proxy.

In [7]:
for name, df in [("fonasa", fonasa), ("suseso", suseso)]:
    key = df[name_fields].fillna("").agg("|".join, axis=1)
    complete = df[name_fields].notna().all(axis=1)
    dup = key[complete].duplicated(keep=False).sum()
    print(f"{name}: {dup:,} of {complete.sum():,} fully-populated records "
          f"({100*dup/complete.sum():.2f}%) share all five identifying fields "
          f"with at least one other record")

fonasa: 0 of 24,149 fully-populated records (0.00%) share all five identifying fields with at least one other record
suseso: 0 of 22,284 fully-populated records (0.00%) share all five identifying fields with at least one other record


None at all here, which tells us these registers are person-level rather than
event-level: each person appears at most once in each file. In real administrative data the answer is frequently very
different: a hospital admissions file has one row per admission, not per
patient.

If your registers are event-based, you have a choice to make before linking:
deduplicate each register first (which is itself a linkage problem, applied
within one file), or link at the event level and aggregate afterwards. Chapter
2.1 covers how that decision follows from the statistical question.

## 5. The three structural prerequisites

Splink, and any comparable library, needs three things that have nothing to do
with data quality and everything to do with shape.

In [8]:
checks = []

# 1. A unique identifier per record, within each register
for name, df in [("fonasa", fonasa), ("suseso", suseso)]:
    ok = df["unique_id"].is_unique and df["unique_id"].notna().all()
    checks.append((f"1. {name}: unique_id present and unique", ok))

# 2. The columns to be compared must have the SAME NAMES in both registers
shared = sorted(set(fonasa.columns) & set(suseso.columns))
checks.append(("2. identifying columns share names across registers",
               all(f in shared for f in name_fields)))

# 3. One row per record, with the comparison fields as columns (tidy)
checks.append(("3. one row per record, fields in columns",
               fonasa.index.is_unique and suseso.index.is_unique))

for label, ok in checks:
    print(f"{'PASS' if ok else 'FAIL'}  {label}")

PASS  1. fonasa: unique_id present and unique
PASS  1. suseso: unique_id present and unique
PASS  2. identifying columns share names across registers
PASS  3. one row per record, fields in columns


The second one catches people out. If one register calls it `nombre` and the
other calls it `first_name`, no library will compare them: renaming is your
job, and it belongs here, in preparation, not buried in a model definition.

## 6. Cleaning and standardisation

Now the substantive part. The goal is to make values that *represent the same
thing* look identical, without making values that represent *different things*
look identical.

Three functions do almost all of the work.

In [9]:
def basic_text_clean(series):
    """Strip, uppercase, and collapse repeated whitespace."""
    return (
        series.astype("string")
        .str.strip()
        .str.upper()
        .str.replace(r"\s+", " ", regex=True)
    )


def remove_accents(value):
    """Drop combining accent marks from a single string."""
    if pd.isna(value):
        return pd.NA
    value = unicodedata.normalize("NFKD", str(value))
    return "".join(c for c in value if not unicodedata.combining(c))


def standardise_name(series):
    """Clean, strip accents, and keep only letters and single spaces."""
    cleaned = basic_text_clean(series)
    cleaned = cleaned.map(remove_accents, na_action="ignore").astype("string")
    cleaned = cleaned.str.replace(r"[^A-ZN ]", "", regex=True)
    return cleaned.str.replace(r"\s+", " ", regex=True).str.strip()


def clean_sex(series):
    """Harmonise sex labels to F / M; anything unrecognised becomes missing."""
    cleaned = basic_text_clean(series)
    return cleaned.replace({
        "MASCULINO": "M", "MALE": "M", "HOMBRE": "M", "M": "M",
        "FEMENINO": "F", "FEMALE": "F", "MUJER": "F", "F": "F",
        "INDETERMINADO": pd.NA, "UNKNOWN": pd.NA, "": pd.NA,
    })

A note on `standardise_name`. Stripping everything that is not a letter is a
deliberately blunt instrument: it collapses `O'MALLEY`, `OMALLEY` and
`O MALLEY` onto the same value, which is usually what you want when the same
person has been entered by three different clerks. It also collapses genuinely
different names in rarer cases. That trade-off is the subject of section 8
below, and it is a decision you should make on your own data rather than
inherit.

Apply exactly the same functions to both registers. Cleaning the two sides
differently is a reliable way to destroy matches you would otherwise have
found.

In [10]:
for df in (fonasa, suseso):
    for col in ["nombre", "ap1", "ap2"]:
        df[f"{col}_clean"] = standardise_name(df[col])
    df["sexo_clean"] = clean_sex(df["sexo"])
    df["nac_clean"] = basic_text_clean(df["nacionalidad"])

clean_fields = ["nombre_clean", "ap1_clean", "ap2_clean", "sexo_clean", "nac_clean"]
fonasa[["nombre", "nombre_clean", "ap1", "ap1_clean", "sexo", "sexo_clean"]].head(5)

,nombre,nombre_clean,ap1,ap1_clean,sexo,sexo_clean
0,NaN,<NA>,RUJES,RUJES,MUJER,F
1,NICOLÁS ALBERTK,NICOLAS ALBERTK,LOBOS,LOBOS,HOMBRE,M
2,JOSEFA DANIELA,JOSEFA DANIELA,RINCON,RINCON,MUJER,F
3,GUIVALDO MOISÉS,GUIVALDO MOISES,HOBRAN,HOBRAN,HOMBRE,M
4,KAREN IVONNE,KAREN IVONNE,GODOY,GODOY,MUJER,F


## 7. What did cleaning actually change?

Do not take it on trust. Count.

In [11]:
rows = []
for name, df in [("fonasa", fonasa), ("suseso", suseso)]:
    for raw, cleaned in [("nombre", "nombre_clean"), ("ap1", "ap1_clean"),
                         ("ap2", "ap2_clean"), ("sexo", "sexo_clean"),
                         ("nacionalidad", "nac_clean")]:
        both = df[raw].notna() & df[cleaned].notna()
        changed = (df.loc[both, raw].str.upper().str.strip() != df.loc[both, cleaned]).sum()
        rows.append({
            "register": name,
            "field": raw,
            "values_changed": changed,
            "pct_of_non_missing": round(100 * changed / both.sum(), 2),
            "distinct_before": df[raw].nunique(),
            "distinct_after": df[cleaned].nunique(),
        })

pd.DataFrame(rows)

,register,field,values_changed,pct_of_non_missing,distinct_before,distinct_after
0,fonasa,nombre,5172,18.14,20698,20190
1,fonasa,ap1,5175,18.14,9951,9549
2,fonasa,ap2,5101,17.93,8468,8136
3,fonasa,sexo,29053,100.00,2,2
4,fonasa,nacionalidad,0,0.00,2,2
5,suseso,nombre,3571,13.89,18532,18005
6,suseso,ap1,3272,12.73,9910,9560
7,suseso,ap2,3923,15.35,11160,10839
8,suseso,sexo,26345,100.00,2,2
9,suseso,nacionalidad,0,0.00,2,2


Three things to read here.

`values_changed` for the name fields, around 13-18%, is mostly accents being
stripped. For `sexo` it is 100%, which is not a data-quality finding: every
value changed because `HOMBRE` and `MUJER` were recoded to `M` and `F`. Recoding
and correction both show up in the same column, so read it alongside what the
function actually does.

`nacionalidad` changed nothing, because its values were already uppercase and
unaccented. Cleaning a field that needs no cleaning is harmless, and cheaper
than checking first.

`distinct_before` versus `distinct_after` is the most informative pair, because
it shows **collapse**: how many previously distinct spellings now share a value.
Around 400 surname spellings in each register merged into another. Every
collapsed pair is a potential match that would otherwise have been missed, and
also a potential false match that has just become possible.

## 8. The standardisation trade-off

There is no neutral amount of cleaning. Standardising more finds more true
matches and creates more false ones. Standardising less does the reverse.

Consider three records with the surname `O'MALLEY`, `OMALLEY` and `O MALLEY`.

- If your data is high quality, and clerks record apostrophes reliably, those
  three spellings probably belong to three different people, and collapsing
  them manufactures false matches.
- If your data is low quality, they are almost certainly the same person
  recorded three ways, and *not* collapsing them loses two matches.

The right amount of standardisation therefore depends on the error rate in
your data, which you have to measure rather than assume. It is worth trying
more than one level and comparing results, which is exactly what the evaluation
chapter makes possible.

You can see the effect directly by comparing a lighter cleaning rule with the
one used above.

In [12]:
light = basic_text_clean(fonasa["ap1"])              # case and spacing only
heavy = fonasa["ap1_clean"]                          # + accents + punctuation

print(f"distinct first surnames, raw           : {fonasa['ap1'].nunique():,}")
print(f"distinct first surnames, light cleaning: {light.nunique():,}")
print(f"distinct first surnames, full cleaning : {heavy.nunique():,}")
print()
print(f"values collapsed by the extra cleaning : {light.nunique() - heavy.nunique():,}")

distinct first surnames, raw           : 9,951
distinct first surnames, light cleaning: 9,950
distinct first surnames, full cleaning : 9,549

values collapsed by the extra cleaning : 401


## 9. A readiness summary

Finally, a compact statement of where the two registers stand. Something like
this belongs in your project documentation: it is the evidence for the design
decisions taken in the next chapters.

In [13]:
summary = pd.DataFrame({
    "check": [
        "Records",
        "Identifying fields available",
        "Records with all identifying fields",
        "Most discriminating field (distinct values)",
        "Least discriminating field (distinct values)",
        "Duplicate-looking records",
        "Unique record identifier",
        "Column names aligned across registers",
    ],
    "fonasa": [
        f"{len(fonasa):,}",
        f"{len(name_fields)}",
        f"{100*fonasa[name_fields].notna().all(axis=1).mean():.1f}%",
        f"nombre_clean ({fonasa['nombre_clean'].nunique():,})",
        f"sexo_clean ({fonasa['sexo_clean'].nunique()})",
        "<0.1%",
        "yes (unique_id)",
        "yes",
    ],
    "suseso": [
        f"{len(suseso):,}",
        f"{len(name_fields)}",
        f"{100*suseso[name_fields].notna().all(axis=1).mean():.1f}%",
        f"nombre_clean ({suseso['nombre_clean'].nunique():,})",
        f"sexo_clean ({suseso['sexo_clean'].nunique()})",
        "<0.1%",
        "yes (unique_id)",
        "yes",
    ],
})
summary

,check,fonasa,suseso
0,Records,"30,000","27,000"
1,Identifying fields available,5,5
2,Records with all identifying fields,80.5%,82.5%
3,Most discriminating field (distinct values),"nombre_clean (20,190)","nombre_clean (18,005)"
4,Least discriminating field (distinct values),sexo_clean (2),sexo_clean (2)
5,Duplicate-looking records,<0.1%,<0.1%
6,Unique record identifier,yes (unique_id),yes (unique_id)
7,Column names aligned across registers,yes,yes


## Where this goes next

The registers are now comparable: same column names, same conventions, known
completeness, known discriminating power.

Two paths lead out of here.

- If the extracts have to cross an institutional boundary before they can be
  linked, [chapter 2.3](anonymisation.md) covers pseudonymisation, and
  what it does and does not protect.
- Otherwise, [chapter 2.4](linkage-approaches.md) starts comparing records, first
  with deterministic rules and then with the Fellegi-Sunter framework.